# 05 · Executive Dashboard, District Map, Alerts & Site Assembly

The final notebook in the series. It:

1. Builds the **statewide executive KPI scorecard** (radar chart + summary numbers)
2. Renders an **interactive Folium map** of districts colored by risk tier
3. Generates a threshold-based **alerts feed** (the same pattern as a real operations dashboard)
4. Assembles a **methodology** reference page summarizing the statistical techniques used across
   Notebooks 02–04
5. Builds the GitHub Pages **landing page** (`docs/index.html`) that ties the whole site together

Run this notebook *last*, after `01`–`04`.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import folium
from pathlib import Path
import json

DATA_DIR = Path("../data")
DOCS_DIR = Path("../docs")
PLOT_TEMPLATE = "plotly_white"

districts = pd.read_csv(DATA_DIR / "districts.csv")
submissions = pd.read_csv(DATA_DIR / "submissions.csv")
risk_scores = pd.read_csv(DATA_DIR / "risk_scores.csv")
idea_indicators = pd.read_csv(DATA_DIR / "idea_indicators.csv")
violations = pd.read_csv(DATA_DIR / "validation_violations.csv")

LATEST_YEAR = "2025-26"
LATEST_MONTH = submissions.period.max()
latest_risk = risk_scores[risk_scores.school_year == LATEST_YEAR].copy()
print(f"Latest school year: {LATEST_YEAR} · Latest reporting month: {LATEST_MONTH}")

Latest school year: 2025-26 · Latest reporting month: 2026-08


## 1. Statewide executive KPI scorecard

In [2]:
kpis = dict(
    n_districts=districts.district_id.nunique(),
    n_regions=districts.region.nunique(),
    statewide_completeness=submissions.completeness_rate.mean(),
    statewide_on_time=submissions.on_time.mean(),
    statewide_error_rate=submissions.validity_error_rate.mean(),
    pct_indicators_met=idea_indicators.met_target.mean(),
    n_meets=int((latest_risk.risk_tier == "Meets Requirements").sum()),
    n_assist=int((latest_risk.risk_tier == "Needs Assistance").sum()),
    n_intervention=int((latest_risk.risk_tier == "Needs Intervention").sum()),
    n_substantial=int((latest_risk.risk_tier == "Needs Substantial Intervention").sum()),
)
for k, v in kpis.items():
    print(f"{k:28s} {v}")

n_districts                  60
n_regions                    5
statewide_completeness       0.9303612673611111
statewide_on_time            0.9055555555555556
statewide_error_rate         0.053553125000000014
pct_indicators_met           0.8430555555555556
n_meets                      37
n_assist                     14
n_intervention               6
n_substantial                3


In [3]:
# Statewide scorecard radar: five quality dimensions normalized 0-100 against targets
radar_metrics = {
    "Completeness":  (submissions.completeness_rate.mean(), 0.98),
    "Timeliness":    (submissions.on_time.mean(), 0.95),
    "Validity":      (1 - submissions.validity_error_rate.mean(), 0.98),
    "Low Duplicates":(1 - min(submissions.duplicate_count.sum() / submissions.records_submitted.sum(), 1), 0.995),
    "Indicators Met":(idea_indicators.met_target.mean(), 0.90),
}
labels = list(radar_metrics.keys())
actual = [radar_metrics[k][0] * 100 for k in labels]
target = [radar_metrics[k][1] * 100 for k in labels]

fig_radar = go.Figure()
fig_radar.add_trace(go.Scatterpolar(r=target + [target[0]], theta=labels + [labels[0]],
                                     name="Target", line=dict(color="#9AA6C3", dash="dot")))
fig_radar.add_trace(go.Scatterpolar(r=actual + [actual[0]], theta=labels + [labels[0]],
                                     name="Statewide actual", fill="toself",
                                     line=dict(color="#12A594"), fillcolor="rgba(18,165,148,0.25)"))
fig_radar.update_layout(template=PLOT_TEMPLATE, height=440, title="Statewide KPI Scorecard",
                         polar=dict(radialaxis=dict(range=[70, 100])))
fig_radar.show()

In [4]:
region_summary = submissions.groupby("region").agg(
    completeness=("completeness_rate", "mean"), on_time=("on_time", "mean"),
    error_rate=("validity_error_rate", "mean")
).round(3)
region_risk = latest_risk.groupby("region").risk_index.mean().round(1)
region_summary["avg_risk_index"] = region_risk
region_summary = region_summary.sort_values("avg_risk_index", ascending=False)

fig_region = px.bar(region_summary.reset_index(), x="region", y="avg_risk_index", color="avg_risk_index",
                     color_continuous_scale=["#12A594", "#E8A33D", "#D64550"],
                     template=PLOT_TEMPLATE, title=f"Average Risk Index by Region ({LATEST_YEAR})",
                     labels={"avg_risk_index": "Avg. risk index (0-100)"})
fig_region.update_layout(height=380, coloraxis_showscale=False)
fig_region.show()
region_summary

,completeness,on_time,error_rate,avg_risk_index
region,,,,
Central NY,0.909,0.884,0.072,21.9
Western NY,0.917,0.886,0.066,21.0
North Country,0.925,0.891,0.061,17.8
NYC Metro,0.938,0.918,0.048,14.7
Capital Region,0.963,0.949,0.021,6.5


## 2. Interactive district map (Folium)

In [5]:
COUNTY_COORDS = {
    "Kings": (40.6782, -73.9442), "Queens": (40.7282, -73.7949), "Bronx": (40.8448, -73.8648),
    "New York": (40.7831, -73.9712), "Richmond": (40.5795, -74.1502), "Nassau": (40.7400, -73.5594),
    "Westchester": (41.1220, -73.7949), "Erie": (42.8864, -78.8784), "Monroe": (43.1610, -77.6109),
    "Niagara": (43.1855, -78.8988), "Chautauqua": (42.2223, -79.2353), "Onondaga": (43.0481, -76.1474),
    "Oneida": (43.2029, -75.4494), "Broome": (42.0987, -75.9180), "Chemung": (42.1009, -76.7938),
    "Albany": (42.6001, -73.9662), "Rensselaer": (42.6803, -73.6899), "Schenectady": (42.8142, -73.9396),
    "Saratoga": (43.0831, -73.7846), "Clinton": (44.6995, -73.4529), "Jefferson": (43.9748, -75.9108),
    "St. Lawrence": (44.5259, -75.1449), "Franklin": (44.5931, -74.3118),
}
TIER_COLOR = {
    "Meets Requirements": "#12A594", "Needs Assistance": "#E8A33D",
    "Needs Intervention": "#D64550", "Needs Substantial Intervention": "#7A1F27",
}

rng = np.random.default_rng(3)
map_df = latest_risk.merge(districts[["district_id", "county", "enrollment_total"]], on="district_id")

m = folium.Map(location=[42.9, -75.5], zoom_start=6, tiles="CartoDB positron")
for _, row in map_df.iterrows():
    base_lat, base_lon = COUNTY_COORDS.get(row.county, (42.9, -75.5))
    lat = base_lat + rng.uniform(-0.12, 0.12)
    lon = base_lon + rng.uniform(-0.12, 0.12)
    popup_html = (f"<b>{row.district_name}</b><br>"
                   f"Region: {row.region}<br>County: {row.county}<br>"
                   f"Risk tier: <b>{row.risk_tier}</b><br>"
                   f"Risk index: {row.risk_index:.1f}<br>"
                   f"Completeness: {row.mean_completeness:.1%}<br>"
                   f"On-time rate: {row.on_time_rate:.1%}")
    folium.CircleMarker(
        location=[lat, lon],
        radius=5 + (row.enrollment_total ** 0.5) / 60,
        color=TIER_COLOR[row.risk_tier], fill=True, fill_color=TIER_COLOR[row.risk_tier],
        fill_opacity=0.85, weight=1,
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(m)

legend_html = '''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white;
     padding: 12px 16px; border-radius: 8px; box-shadow: 0 4px 14px rgba(0,0,0,0.18); font-family: Inter, sans-serif; font-size: 12px;">
  <b>Risk tier</b><br>
  <span style="color:#12A594;">●</span> Meets Requirements<br>
  <span style="color:#E8A33D;">●</span> Needs Assistance<br>
  <span style="color:#D64550;">●</span> Needs Intervention<br>
  <span style="color:#7A1F27;">●</span> Needs Substantial Intervention
</div>'''
m.get_root().html.add_child(folium.Element(legend_html))

map_path = DOCS_DIR / "dashboard" / "_folium_map.html"
m.save(str(map_path))
print("Folium map saved to", map_path)
m

Folium map saved to ../docs/dashboard/_folium_map.html


## 3. Threshold-based alerts feed

In [6]:
alerts = []

# Critical: latest-month rejected submissions
crit_sub = submissions[(submissions.period == LATEST_MONTH) & (submissions.status == "Rejected")]
for _, r in crit_sub.iterrows():
    dname = districts.set_index("district_id").loc[r.district_id, "district_name"]
    alerts.append(dict(level="crit", title=f"{dname} — {r.collection} submission REJECTED",
                        detail=f"Completeness {r.completeness_rate:.0%}, validity error {r.validity_error_rate:.0%} for {r.period}"))

# Critical: districts in the worst risk tier
for _, r in latest_risk[latest_risk.risk_tier == "Needs Substantial Intervention"].iterrows():
    alerts.append(dict(level="crit", title=f"{r.district_name} — Needs Substantial Intervention",
                        detail=f"Risk index {r.risk_index:.1f}/100 · {r.region} · {LATEST_YEAR}"))

# Warning: districts in Needs Intervention
for _, r in latest_risk[latest_risk.risk_tier == "Needs Intervention"].iterrows():
    alerts.append(dict(level="warn", title=f"{r.district_name} — Needs Intervention",
                        detail=f"Risk index {r.risk_index:.1f}/100 · {r.region} · {LATEST_YEAR}"))

# Warning: latest-month "Accepted with Warnings"
warn_sub = submissions[(submissions.period == LATEST_MONTH) & (submissions.status == "Accepted with Warnings")]
for _, r in warn_sub.head(6).iterrows():
    dname = districts.set_index("district_id").loc[r.district_id, "district_name"]
    alerts.append(dict(level="warn", title=f"{dname} — {r.collection} accepted with warnings",
                        detail=f"Completeness {r.completeness_rate:.0%}, on-time: {'Yes' if r.on_time else 'No'}"))

alerts_df = pd.DataFrame(alerts)
n_crit = (alerts_df.level == "crit").sum()
n_warn = (alerts_df.level == "warn").sum()
print(f"{len(alerts_df)} alerts generated for {LATEST_MONTH} / {LATEST_YEAR}: {n_crit} critical, {n_warn} warning")
alerts_df.head(10)

59 alerts generated for 2026-08 / 2025-26: 47 critical, 12 warning


,level,title,detail
0,crit,Foxglen CSD — SIRS Enrollment submission REJECTED,"Completeness 73%, validity error 21% for 2026-08"
1,crit,Foxglen CSD — SIRS Assessment submission REJECTED,"Completeness 55%, validity error 27% for 2026-08"
2,crit,Foxglen CSD — SIRS Special Education submissio...,"Completeness 81%, validity error 35% for 2026-08"
3,crit,Foxglen CSD — IDEA 618 Exiting submission REJE...,"Completeness 88%, validity error 33% for 2026-08"
4,crit,Foxglen CSD — IDEA Maintenance of Effort submi...,"Completeness 72%, validity error 26% for 2026-08"
5,crit,Foxglen CSD — Student Attendance (EMSC) submis...,"Completeness 72%, validity error 33% for 2026-08"
6,crit,Southport CSD — IDEA 618 Child Count submissio...,"Completeness 88%, validity error 16% for 2026-08"
7,crit,Goldenside CSD — SIRS Enrollment submission RE...,"Completeness 64%, validity error 33% for 2026-08"
8,crit,Goldenside CSD — SIRS Assessment submission RE...,"Completeness 59%, validity error 35% for 2026-08"
9,crit,Goldenside CSD — SIRS Special Education submis...,"Completeness 55%, validity error 35% for 2026-08"


## 4. Site templates (shared nav / page shell / KPI cards)

In [7]:
NAV_PAGES = json.load(open("../config/nav_pages.json"))

def render_nav(active):
    links = "\n".join(f'<a href="{href}" class="{"active" if key == active else ""}">{label}</a>'
                       for key, label, href in NAV_PAGES)
    return f'''<nav class="site-nav"><div class="nav-inner">
      <div class="brand"><span class="dot"></span> NYSED Data Integrity Platform</div>
      <div class="links">{links}</div></div></nav>'''

def page_shell(title, eyebrow, description, body_html, active, filename, extra_head=""):
    html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title} · NYSED Data Integrity Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="../assets/style.css">
{extra_head}
</head><body>
{render_nav(active)}
<header class="dash-header"><div class="dash-header-inner">
  <div class="eyebrow">{eyebrow}</div><h1>{title}</h1><p>{description}</p>
</div></header>
<main class="dash-body">{body_html}</main>
<footer class="site-footer">Synthetic demonstration data · NYSED Data Integrity &amp; Risk Monitoring Platform ·
  <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">Source code</a></footer>
</body></html>'''
    out = DOCS_DIR / "dashboard" / filename
    out.write_text(html)
    print("wrote", out)

def kpi_card(label, value, delta=None, tone="good", warn=False, crit=False):
    cls = "kpi-card" + (" crit" if crit else " warn" if warn else "")
    delta_html = f'<div class="kpi-delta {tone}">{delta}</div>' if delta else ""
    return f'<div class="{cls}"><div class="kpi-label">{label}</div><div class="kpi-value">{value}</div>{delta_html}</div>'

def fig_div(f, div_id, include_js=False):
    return pio.to_html(f, include_plotlyjs="inline" if include_js else False, full_html=False, div_id=div_id)

## 5. Build `executive-summary.html`

In [8]:
kpi_html = f'''<div class="kpi-row">
  {kpi_card("Districts Monitored", f"{kpis['n_districts']}", f"{kpis['n_regions']} regions")}
  {kpi_card("Statewide Completeness", f"{kpis['statewide_completeness']:.1%}", "Target ≥ 98%",
             tone="good" if kpis['statewide_completeness']>=0.98 else "bad")}
  {kpi_card("On-Time Submission Rate", f"{kpis['statewide_on_time']:.1%}", "Target ≥ 95%",
             tone="good" if kpis['statewide_on_time']>=0.95 else "bad")}
  {kpi_card("IDEA Indicators Met", f"{kpis['pct_indicators_met']:.1%}", "Target ≥ 90%",
             tone="good" if kpis['pct_indicators_met']>=0.90 else "bad")}
</div>'''

tier_row = f'''<div class="kpi-row">
  {kpi_card("Meets Requirements", kpis['n_meets'], tone="good")}
  {kpi_card("Needs Assistance", kpis['n_assist'], warn=True, tone="bad")}
  {kpi_card("Needs Intervention", kpis['n_intervention'], warn=True, tone="bad")}
  {kpi_card("Needs Substantial Intervention", kpis['n_substantial'], crit=True, tone="bad")}
</div>'''

fig_region_small = px.bar(region_summary.reset_index(), x="region", y="avg_risk_index", color="avg_risk_index",
                     color_continuous_scale=["#12A594", "#E8A33D", "#D64550"], template=PLOT_TEMPLATE)
fig_region_small.update_layout(height=340, coloraxis_showscale=False, margin=dict(t=20,l=10,r=10,b=10))

fig_radar.update_layout(margin=dict(t=20,l=20,r=20,b=20))

body = kpi_html + f'''
<h3 style="margin-top:28px;">Risk tier distribution — {LATEST_YEAR}</h3>
{tier_row}
<div class="chart-row">
  <div class="chart-panel"><h3>Statewide KPI scorecard</h3>
    <div class="chart-note">All five dimensions vs. institutional targets (dotted).</div>
    {fig_div(fig_radar, "fig-radar", include_js=True)}</div>
  <div class="chart-panel"><h3>Average risk index by region</h3>
    <div class="chart-note">{LATEST_YEAR} · lower is better.</div>
    {fig_div(fig_region_small, "fig-region")}</div>
</div>
<div class="chart-panel"><h3>Regional summary table</h3>
  <table class="kpi-table"><thead><tr><th>Region</th><th>Completeness</th><th>On-Time</th><th>Error Rate</th><th>Avg. Risk Index</th></tr></thead><tbody>
  {"".join(f"<tr><td>{reg}</td><td class='target'>{row.completeness:.1%}</td><td class='target'>{row.on_time:.1%}</td><td class='target'>{row.error_rate:.1%}</td><td class='target'>{row.avg_risk_index:.1f}</td></tr>" for reg, row in region_summary.iterrows())}
  </tbody></table>
</div>
'''

page_shell(title="Executive Summary", eyebrow="Notebook 05",
           description="Statewide KPI scorecard, risk-tier distribution, and regional breakdown for the current reporting period.",
           body_html=body, active="executive-summary.html", filename="executive-summary.html")

wrote ../docs/dashboard/executive-summary.html


## 6. Build `district-map.html`

In [9]:
map_body = f'''
<div class="chart-panel">
  <h3>District risk map — {LATEST_YEAR}</h3>
  <div class="chart-note">Marker size ≈ enrollment; color = risk tier. Positions are jittered within each county for synthetic districts (not real geocoding).</div>
  <iframe src="_folium_map.html" style="width:100%; height:640px; border:0; border-radius:8px;"></iframe>
</div>
'''
page_shell(title="District Map", eyebrow="Notebook 05",
           description="Interactive statewide map of synthetic districts colored by OSEP-style risk determination tier.",
           body_html=map_body, active="district-map.html", filename="district-map.html")

wrote ../docs/dashboard/district-map.html


## 7. Build `alerts.html`

In [10]:
def alert_item(a):
    badge = "CRITICAL" if a["level"] == "crit" else "WARNING"
    return f'''<div class="alert-item {a["level"]}">
      <span class="alert-badge {a["level"]}">{badge}</span>
      <div class="alert-text"><strong>{a["title"]}</strong><br><span>{a["detail"]}</span></div>
    </div>'''

alerts_kpi = f'''<div class="kpi-row">
  {kpi_card("Critical Alerts", n_crit, crit=True, tone="bad")}
  {kpi_card("Warning Alerts", n_warn, warn=True, tone="bad")}
  {kpi_card("Reporting Period", LATEST_MONTH)}
  {kpi_card("School Year", LATEST_YEAR)}
</div>'''

alerts_body = alerts_kpi + '<div class="chart-panel"><h3>Active alerts</h3>' + \
    "".join(alert_item(a) for a in alerts_df.sort_values("level").to_dict("records")) + '</div>'

page_shell(title="Alerts", eyebrow="Notebook 05",
           description=f"Threshold-based critical and warning alerts across all districts for {LATEST_MONTH} ({LATEST_YEAR}).",
           body_html=alerts_body, active="alerts.html", filename="alerts.html")

wrote ../docs/dashboard/alerts.html


## 8. Build `methodology.html`

In [11]:
METHOD_SECTIONS = [
    ("01", "Data Quality Dimensions", [
        "Completeness — missingness rate per field/collection, with pattern analysis (MCAR vs. MAR vs. MNAR)",
        "Validity — domain, range, and format checks against code lists and business rules",
        "Consistency — cross-field logic and referential integrity checks",
        "Accuracy — agreement rate against an audited sample, summarized with Cohen's kappa",
        "Uniqueness — duplicate detection via probabilistic record linkage / fuzzy matching",
        "Timeliness — on-time submission rate and lag-time distribution",
    ]),
    ("02", "Statistical Process Control", [
        "Shewhart p-charts for error / rejection / late-submission rates, with 3σ control limits",
        "CUSUM and EWMA charts to catch slow drift before it crosses a hard threshold",
        "STL-style decomposition to separate seasonal reporting cycles from genuine anomalies",
    ]),
    ("03", "Anomaly & Outlier Detection", [
        "Z-score and modified Z-score (MAD-based) for univariate outliers",
        "IQR fences as a robust, distribution-free alternative",
        "Isolation Forest for multivariate anomalies across several quality metrics at once",
        "Benford's Law leading-digit test as a fabrication / manipulation screen (with caveats for stratified data)",
    ]),
    ("04", "Record Linkage / Duplicate Detection", [
        "Blocking (e.g., by district) to make pairwise comparison tractable",
        "Jaro-Winkler string similarity on name fields",
        "Fellegi-Sunter-style weighted composite match score",
        "Threshold selection via precision / recall / F1 sweep",
    ]),
    ("05", "Predictive Risk Modeling", [
        "Logistic regression for an interpretable, coefficient-level risk model",
        "Random forest / gradient boosting for higher-accuracy classification with feature importance",
        "ROC / AUC evaluation on a held-out test set",
        "Care taken to avoid circularity: predictors must be independent of the label's own construction",
    ]),
    ("06", "Small-Area Estimation", [
        "Empirical Bayes (method-of-moments) shrinkage of small-district rates toward the regional mean",
        "Funnel plots to visualize the precision/denominator relationship",
        "Directly analogous to small-area disease-mapping techniques in spatial epidemiology",
    ]),
    ("07", "Composite Risk Index", [
        "Standardize (z-score) each quality metric, with sign-flipping so higher always means worse",
        "Weighted linear combination into a single 0-100 risk index",
        "Quantile-based tiering into four OSEP-style risk determination categories",
    ]),
]

def method_block(num, title, items):
    lis = "".join(f"<li>{i}</li>" for i in items)
    return f'''<div class="method-block">
      <div class="step-num">{num}</div>
      <h3>{title}</h3>
      <ul>{lis}</ul>
    </div>'''

method_body = "".join(method_block(*s) for s in METHOD_SECTIONS)
method_body = f'<p style="max-width:760px;">This platform combines seven families of statistical technique across the four analysis notebooks. Each is demonstrated on the synthetic dataset with real, runnable code.</p>' + method_body

page_shell(title="Methodology", eyebrow="Reference",
           description="The statistical techniques underlying data quality assessment, validation, and risk scoring across this platform.",
           body_html=method_body, active="methodology.html", filename="methodology.html")

wrote ../docs/dashboard/methodology.html


## 9. Build the site landing page (`docs/index.html`)

In [12]:
gauge_score = round(100 - latest_risk.risk_index.mean(), 1)   # "Data Trust Score" = inverse of avg risk index

def gauge_svg(score, size=200):
    r = 78
    circumference = 2 * np.pi * r
    offset = circumference * (1 - score / 100)
    cx = cy = size / 2
    return f'''<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}">
      <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke-width="14" class="gauge-arc-bg"/>
      <circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke-width="14" class="gauge-arc-fg"
        stroke-dasharray="{circumference:.1f}" stroke-dashoffset="{offset:.1f}"
        transform="rotate(-90 {cx} {cy})"/>
      <text x="{cx}" y="{cy-2}" text-anchor="middle" class="gauge-score">{score:.0f}</text>
      <text x="{cx}" y="{cy+22}" text-anchor="middle" class="gauge-sub">/ 100</text>
    </svg>'''

DASH_CARDS = [
    ("📊", "Executive Summary", "Statewide KPI scorecard with risk-tier breakdown and regional comparison.", "dashboard/executive-summary.html"),
    ("🔍", "Data Quality", "Completeness, validity, consistency, accuracy, and a timeliness control chart.", "dashboard/data-quality.html"),
    ("🧩", "Validation & Duplicates", "Rule engine results plus Jaro-Winkler / Fellegi-Sunter duplicate detection.", "dashboard/validation.html"),
    ("⚠️", "Risk Assessment", "CUSUM/EWMA drift, outlier detection, Benford's Law, predictive risk model, Bayes shrinkage.", "dashboard/risk-assessment.html"),
    ("🗺️", "District Map", "Interactive Folium map of districts colored by OSEP-style risk tier.", "dashboard/district-map.html"),
    ("🔔", "Alerts", "Threshold-based critical and warning alerts for the current reporting period.", "dashboard/alerts.html"),
    ("📚", "Methodology", "Reference summary of every statistical technique used across the platform.", "dashboard/methodology.html"),
]

cards_html = "".join(
    f'''<a class="card" href="{href}"><div class="icon">{icon}</div><h3>{title}</h3><p>{desc}</p>
        <span class="tag">Open →</span></a>'''
    for icon, title, desc, href in DASH_CARDS
)

KPI_ROWS = [
    ("Data Completeness Rate", "Quality", "≥ 98%"),
    ("Validity Error Rate", "Quality", "≤ 2%"),
    ("On-Time Submission Rate", "Operational", "≥ 95%"),
    ("Duplicate Record Rate", "Quality", "≤ 0.5%"),
    ("IDEA Indicator Compliance", "Compliance", "≥ 90%"),
    ("Cohen's κ (Accuracy Audit)", "Quality", "≥ 0.80"),
    ("Composite Risk Index", "Risk", "≤ 40 / 100"),
    ("Isolation Forest Anomaly Rate", "Risk", "≤ 8%"),
]
PILL_CLASS = {"Quality": "pill-quality", "Operational": "pill-ops", "Compliance": "pill-compliance", "Risk": "pill-risk"}
kpi_table_rows = "".join(
    f"<tr><td>{name}</td><td><span class='pill {PILL_CLASS[cat]}'>{cat}</span></td><td class='target'>{target}</td></tr>"
    for name, cat, target in KPI_ROWS
)

REGION_CARDS = "".join(
    f'''<div class="region-card"><h4>{region}</h4><p>{", ".join(counties)}</p></div>'''
    for region, counties in {
        "NYC Metro": ["Kings","Queens","Bronx","New York","Richmond","Nassau","Westchester"],
        "Western NY": ["Erie","Monroe","Niagara","Chautauqua"],
        "Central NY": ["Onondaga","Oneida","Broome","Chemung"],
        "Capital Region": ["Albany","Rensselaer","Schenectady","Saratoga"],
        "North Country": ["Clinton","Jefferson","St. Lawrence","Franklin"],
    }.items()
)

index_html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="description" content="NYSED Data Integrity &amp; Risk Monitoring Platform — synthetic statistical demonstration of data quality, validation, and risk assessment for state education data collections.">
<title>NYSED Data Integrity &amp; Risk Monitoring Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="assets/style.css">
</head><body>
<nav class="site-nav"><div class="nav-inner">
  <div class="brand"><span class="dot"></span> NYSED Data Integrity Platform</div>
  <div class="links">
    <a href="#features">Features</a>
    <a href="dashboard/executive-summary.html">Dashboard</a>
    <a href="#kpis">KPIs</a>
    <a href="#quick-start">Quick Start</a>
    <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">GitHub</a>
  </div>
</div></nav>

<section class="hero">
  <div class="hero-inner">
    <div>
      <div class="eyebrow">New York State · Data Governance &amp; Compliance Analytics</div>
      <h1>Data Integrity &amp; Risk Monitoring Platform</h1>
      <p class="lede">A statistical framework for evaluating <b>completeness, validity, consistency,
        accuracy, and timeliness</b> across state education data collections — with control charts,
        anomaly detection, duplicate/record-linkage matching, and a predictive district risk model,
        built entirely in Python notebooks on simulated data.</p>
      <div class="cta-row">
        <a class="btn btn-primary" href="dashboard/executive-summary.html">Open Dashboard</a>
        <a class="btn btn-ghost" href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">View on GitHub</a>
        <a class="btn btn-ghost" href="#quick-start">Run locally</a>
      </div>
      <div class="disclaimer">⚠️ <b>Demonstration purpose only.</b> All district names, enrollment
        figures, submission records, and risk determinations are entirely simulated and do not
        represent any real school district, student, or New York State agency data.</div>
    </div>
    <div class="gauge-wrap">
      {gauge_svg(gauge_score)}
      <div class="gauge-label">Statewide Data Trust Score</div>
    </div>
  </div>
</section>

<div class="stat-strip"><div class="stat-grid">
  <div class="stat-cell"><div class="num">{kpis['n_districts']}</div><div class="lbl">Districts</div></div>
  <div class="stat-cell"><div class="num">{kpis['n_regions']}</div><div class="lbl">Regions</div></div>
  <div class="stat-cell"><div class="num">8</div><div class="lbl">Collections tracked</div></div>
  <div class="stat-cell"><div class="num">7</div><div class="lbl">Dashboard pages</div></div>
  <div class="stat-cell"><div class="num">{kpis['statewide_completeness']:.0%}</div><div class="lbl">Mean completeness</div></div>
</div></div>

<section class="section" id="features">
  <div class="section-head">
    <div class="eyebrow">Dashboard</div>
    <h2>Seven focused views</h2>
    <p>Built on five Jupyter notebooks — synthetic data generation, data quality assessment,
       validation &amp; duplicate detection, risk assessment, and executive/dashboard assembly.</p>
  </div>
  <div class="card-grid">{cards_html}</div>
</section>

<section class="section" id="kpis">
  <div class="section-head">
    <div class="eyebrow">KPI dimensions</div>
    <h2>Balanced metrics across quality, operations, compliance &amp; risk</h2>
  </div>
  <table class="kpi-table"><thead><tr><th>KPI</th><th>Category</th><th>Target</th></tr></thead>
  <tbody>{kpi_table_rows}</tbody></table>
</section>

<section class="section">
  <div class="section-head">
    <div class="eyebrow">Coverage</div>
    <h2>Regions &amp; counties</h2>
  </div>
  <div class="region-grid">{REGION_CARDS}</div>
</section>

<section class="section" id="quick-start">
  <div class="section-head">
    <div class="eyebrow">Quick start</div>
    <h2>Run the notebooks locally</h2>
    <p>Clone the repository, install dependencies, and run the notebooks in order (01 → 05).
       Each notebook writes its outputs to <code>data/</code> and <code>docs/dashboard/</code>.</p>
  </div>
  <div class="code-block"><span class="cmt"># clone &amp; install</span><br>
git clone https://github.com/zia207/NYSED-Data-Integrity.git<br>
cd NYSED-Data-Integrity<br>
pip install -r requirements.txt<br><br>
<span class="cmt"># run notebooks in order</span><br>
jupyter nbconvert --to notebook --execute --inplace notebooks/01_data_generation.ipynb<br>
jupyter nbconvert --to notebook --execute --inplace notebooks/02_data_quality_assessment.ipynb<br>
jupyter nbconvert --to notebook --execute --inplace notebooks/03_validation_engine.ipynb<br>
jupyter nbconvert --to notebook --execute --inplace notebooks/04_risk_assessment.ipynb<br>
jupyter nbconvert --to notebook --execute --inplace notebooks/05_executive_dashboard.ipynb<br><br>
<span class="cmt"># then open docs/index.html, or serve the docs/ folder via GitHub Pages</span></div>
</section>

<footer class="site-footer">
  © 2026 Zia Ahmed, Uppata Analytics · Synthetic demonstration data · Statistical methodology
  spans SPC, anomaly detection, record linkage, predictive modeling &amp; small-area estimation ·
  <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">Source code</a>
</footer>
</body></html>'''

(DOCS_DIR / "index.html").write_text(index_html)
print("wrote", DOCS_DIR / "index.html")

wrote ../docs/index.html


## 10. Final file inventory

In [13]:
print("docs/ contents:")
for p in sorted(DOCS_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(DOCS_DIR)}  ({p.stat().st_size/1024:.1f} KB)")

docs/ contents:
  assets/style.css  (11.7 KB)
  dashboard/_folium_map.html  (75.2 KB)
  dashboard/alerts.html  (17.3 KB)
  dashboard/data-quality.html  (4772.7 KB)
  dashboard/district-map.html  (1.9 KB)
  dashboard/executive-summary.html  (4750.5 KB)
  dashboard/methodology.html  (4.9 KB)
  dashboard/risk-assessment.html  (4775.5 KB)
  dashboard/validation.html  (6527.8 KB)
  index.html  (8.9 KB)
